In [1]:

from pathlib import Path
import pandas as pd

# =========================
# 1. Locate project root
# =========================

relative_path = Path(
    "data/processed/complaints_nlp.pkl"
)

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find {relative_path} from {Path.cwd()}."
    )

# =========================
# 2. Load NLP checkpoint
# =========================

complaints_nlp = pd.read_pickle(
    project_root / relative_path
)

print("Project root:", project_root)
print("Shape:", complaints_nlp.shape)
print()
print("Columns:")
print(complaints_nlp.columns.tolist())

Project root: d:\data mining\e_waste_topic_modeling
Shape: (130738, 7)

Columns:
['asin', 'reviewText', 'summary', 'overall', 'review_date', 'year', 'final_tokens']


In [2]:
REMOVE_TOKENS = {"s", "'s", "’s"}

complaints_nlp["final_tokens"] = (
    complaints_nlp["final_tokens"]
    .map(lambda tokens: [
        token for token in tokens
        if token.lower() not in REMOVE_TOKENS
    ])
)

print("Cleaning completed.")

Cleaning completed.


In [3]:
remaining_unwanted = complaints_nlp["final_tokens"].map(
    lambda tokens: any(
        token.lower() in REMOVE_TOKENS
        for token in tokens
    )
).sum()

print("Reviews:", len(complaints_nlp))
print("Remaining unwanted tokens:", remaining_unwanted)

Reviews: 130738
Remaining unwanted tokens: 0


In [4]:
complaints_nlp["lda_text"] = (
    complaints_nlp["final_tokens"]
    .map(lambda tokens: " ".join(tokens))
)

print("lda_text created.")

lda_text created.


In [5]:
remaining_s = complaints_nlp["lda_text"].str.contains(
    r"\bs\b", regex=True
).sum()

remaining_possessive_s = complaints_nlp["lda_text"].str.contains(
    r"(?<!\w)['’]s\b", regex=True
).sum()

print("Documents containing standalone 's':", remaining_s)
print("Documents containing possessive 's:", remaining_possessive_s)

Documents containing standalone 's': 402
Documents containing possessive 's: 0


In [6]:
bad_tokens = []

for tokens in complaints_nlp["final_tokens"]:
    for token in tokens:
        if isinstance(token, str) and token.strip().lower() in {"s", "'s", "’s"}:
            bad_tokens.append(repr(token))

print("Remaining problematic tokens:", len(bad_tokens))
print("Examples:", bad_tokens[:30])

Remaining problematic tokens: 0
Examples: []


In [7]:

output_path = (
    project_root
    / "data"
    / "processed"
    / "complaints_nlp_clean.pkl"
)

complaints_nlp.to_pickle(output_path)

print("Saved:", output_path)
print("Shape:", complaints_nlp.shape)

Saved: d:\data mining\e_waste_topic_modeling\data\processed\complaints_nlp_clean.pkl
Shape: (130738, 8)


In [8]:
from pathlib import Path
import pandas as pd

relative_path = Path(
    "data/processed/complaints_nlp_clean.pkl"
)

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find {relative_path} from {Path.cwd()}."
    )

complaints_nlp = pd.read_pickle(
    project_root / relative_path
)

print("Project root:", project_root)
print("Shape:", complaints_nlp.shape)
print("Columns:", complaints_nlp.columns.tolist())

Project root: d:\data mining\e_waste_topic_modeling
Shape: (130738, 8)
Columns: ['asin', 'reviewText', 'summary', 'overall', 'review_date', 'year', 'final_tokens', 'lda_text']


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b\w{2,}\b",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.95,
    max_features=30000
)

X = vectorizer.fit_transform(
    complaints_nlp["lda_text"]
)

feature_names = vectorizer.get_feature_names_out()

print("DTM shape:", X.shape)
print("Number of documents:", X.shape[0])
print("Number of features:", X.shape[1])
print("Standalone 's' in vocabulary:", "s" in vectorizer.vocabulary_)

DTM shape: (130738, 30000)
Number of documents: 130738
Number of features: 30000
Standalone 's' in vocabulary: False


In [10]:
feature_names = vectorizer.get_feature_names_out()

print("First 100 features:")
print(feature_names[:100])

First 100 features:
['00' '00 not' '00 phone' '000' '000 mah' '000mah' '01' '02' '03' '04'
 '05' '06' '07' '08' '09' '0a' '10' '10 00' '10 000' '10 000mah' '10 10'
 '10 12' '10 14' '10 15' '10 16' '10 20' '10 2016' '10 battery' '10 buck'
 '10 cable' '10 case' '10 charge' '10 day' '10 dollar' '10 foot' '10 ft'
 '10 get' '10 hour' '10 inch' '10 min' '10 minute' '10 month' '10 not'
 '10 second' '10 time' '10 year' '100' '100 charge' '100 not'
 '100 percent' '100 phone' '100 sure' '100 time' '1000' '10000' '10000mah'
 '1020' '1080p' '10ft' '10th' '10x' '11' '11 15' '11 16' '11 hour'
 '11 month' '110' '12' '12 15' '12 24' '12 day' '12 hour' '12 month'
 '12 volt' '12 year' '120' '1200' '128gb' '12v' '13' '13 month' '130' '14'
 '14 99' '14 day' '14 month' '15' '15 20' '15 day' '15 min' '15 minute'
 '15 month' '15 not' '15 second' '15 year' '150' '1500' '1520' '16'
 '16 hour']


In [11]:
important_phrases = [
    "battery life",
    "charging port",
    "touch screen",
    "power button",
    "usb cable",
    "battery drain",
    "screen protector",
]

for phrase in important_phrases:
    print(
        f"{phrase:20} :",
        phrase in vectorizer.vocabulary_
    )

battery life         : True
charging port        : True
touch screen         : True
power button         : True
usb cable            : True
battery drain        : True
screen protector     : True


In [12]:
import numpy as np

# Reproducible random sample for LDA model selection
rng = np.random.default_rng(42)

sample_size = min(50000, X.shape[0])

sample_indices = rng.choice(
    X.shape[0],
    size=sample_size,
    replace=False
)

X_sample = X[sample_indices]

print("Full DTM shape   :", X.shape)
print("Sample DTM shape :", X_sample.shape)

Full DTM shape   : (130738, 30000)
Sample DTM shape : (50000, 30000)


In [13]:
from sklearn.decomposition import LatentDirichletAllocation

lda_5 = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="batch",
    max_iter=20,
    evaluate_every=-1,
    n_jobs=-1
)

lda_5.fit(X_sample)

print("LDA training completed.")
print("Number of topics:", lda_5.n_components)
print("Iterations:", lda_5.n_iter_)
print("Perplexity:", round(lda_5.perplexity(X_sample), 2))

LDA training completed.
Number of topics: 5
Iterations: 20
Perplexity: 3041.89


In [14]:
feature_names = vectorizer.get_feature_names_out()

def show_topics(model, feature_names, n_words=15):
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_words:][::-1]
        top_words = feature_names[top_indices]

        print(f"Topic {topic_idx}:")
        print(", ".join(top_words))
        print()
        
show_topics(
    lda_5,
    feature_names,
    n_words=15
)

Topic 0:
screen, not, protector, screen protector, phone, edge, glass, one, get, bubble, crack, cover, use, go, like

Topic 1:
charge, not, work, charger, cable, use, month, one, plug, phone, not work, last, week, usb, stop

Topic 2:
case, not, phone, fit, look, like, use, plastic, break, get, would, make, good, button, back

Topic 3:
not, product, buy, one, get, would, return, money, work, item, back, even, review, send, waste

Topic 4:
phone, not, use, battery, get, work, no, time, would, good, go, call, one, make, like



In [15]:
lda_8 = LatentDirichletAllocation(
    n_components=8,
    random_state=42,
    learning_method="batch",
    max_iter=20,
    evaluate_every=-1,
    n_jobs=-1
)

lda_8.fit(X_sample)

print("LDA training completed.")
print("Number of topics:", lda_8.n_components)
print("Iterations:", lda_8.n_iter_)
print("Perplexity:", round(lda_8.perplexity(X_sample), 2))

LDA training completed.
Number of topics: 8
Iterations: 20
Perplexity: 2970.42


In [16]:
show_topics(
    lda_8,
    feature_names,
    n_words=15
)

Topic 0:
screen, protector, not, screen protector, edge, glass, phone, get, one, bubble, crack, use, cover, go, put

Topic 1:
work, not, month, break, week, one, last, buy, day, use, not work, money, waste, stop, two

Topic 2:
case, phone, not, look, like, get, plastic, button, back, case not, hard, cover, use, make, drop

Topic 3:
not, product, get, one, return, back, send, buy, review, would, receive, amazon, no, order, item

Topic 4:
phone, not, use, get, work, call, no, good, time, go, would, like, problem, one, make

Topic 5:
charge, battery, not, charger, phone, cable, work, plug, use, usb, get, would, port, power, charge phone

Topic 6:
not, fit, would, not fit, like, look, phone, cheap, would not, buy, recommend, make, good, not recommend, well

Topic 7:
phone, not, use, hold, clip, would, mount, car, one, make, well, belt, fall, place, product



In [17]:
lda_10 = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method="batch",
    max_iter=20,
    evaluate_every=-1,
    n_jobs=-1
)

lda_10.fit(X_sample)

print("LDA training completed.")
print("Number of topics:", lda_10.n_components)
print("Iterations:", lda_10.n_iter_)
print("Perplexity:", round(lda_10.perplexity(X_sample), 2))

LDA training completed.
Number of topics: 10
Iterations: 20
Perplexity: 2978.54


In [18]:
show_topics(
    lda_10,
    feature_names,
    n_words=15
)

Topic 0:
screen, protector, not, screen protector, glass, edge, phone, get, one, bubble, cover, use, go, try, make

Topic 1:
work, not, not work, money, waste, week, break, month, buy, waste money, stop, last, good, stop work, quality

Topic 2:
case, phone, not, look, drop, get, break, crack, day, week, color, come, protect, use, start

Topic 3:
not, product, get, return, back, send, would, receive, review, item, no, buy, could, order, amazon

Topic 4:
phone, not, use, get, call, work, good, no, go, time, would, like, sound, make, problem

Topic 5:
charge, charger, not, phone, battery, cable, plug, use, usb, work, port, get, charge phone, not charge, power

Topic 6:
not, fit, would, not fit, like, phone, look, would not, recommend, not recommend, well, buy, stay, cheap, one

Topic 7:
phone, not, hold, use, clip, mount, car, would, belt, make, fall, holder, place, well, one

Topic 8:
one, buy, month, battery, use, new, day, last, not, year, purchase, time, get, update, two

Topic 9:
cas

In [19]:
from sklearn.decomposition import LatentDirichletAllocation

lda_final = LatentDirichletAllocation(
    n_components=8,
    random_state=42,
    learning_method="batch",
    max_iter=20,
    evaluate_every=-1,
    n_jobs=-1
)

lda_final.fit(X)

print("LDA final training completed.")
print("Number of topics:", lda_final.n_components)
print("Iterations:", lda_final.n_iter_)
print("Perplexity:", round(lda_final.perplexity(X), 2))

LDA final training completed.
Number of topics: 8
Iterations: 20
Perplexity: 2743.3


In [20]:
show_topics(
    lda_final,
    feature_names,
    n_words=15
)

Topic 0:
screen, protector, not, screen protector, glass, edge, phone, get, one, bubble, crack, cover, use, go, would

Topic 1:
work, month, not, break, week, one, last, buy, use, day, first, not work, two, stop, start

Topic 2:
case, phone, not, look, like, button, get, plastic, back, drop, hard, case not, cover, use, make

Topic 3:
not, product, one, get, return, review, back, send, receive, would, amazon, order, item, no, give

Topic 4:
phone, not, use, get, work, call, no, good, would, time, go, like, one, problem, sound

Topic 5:
charge, not, battery, charger, phone, cable, work, plug, use, usb, would, get, port, device, power

Topic 6:
not, fit, cheap, not fit, money, look, like, buy, waste, would, phone, quality, good, waste money, get

Topic 7:
not, phone, hold, use, clip, would, mount, make, well, fall, work, car, place, one, belt



In [21]:
doc_topic = lda_final.transform(X)

print("Document-topic matrix shape:", doc_topic.shape)

Document-topic matrix shape: (130738, 8)


In [22]:
import numpy as np

dominant_topic = np.argmax(doc_topic, axis=1)
topic_probability = np.max(doc_topic, axis=1)

print("Number of reviews:", len(dominant_topic))
print("Number of topics:", doc_topic.shape[1])
print("Average dominant-topic probability:", round(topic_probability.mean(), 4))

Number of reviews: 130738
Number of topics: 8
Average dominant-topic probability: 0.6389


In [23]:
topic_counts = np.bincount(
    dominant_topic,
    minlength=lda_final.n_components
)

for topic_id, count in enumerate(topic_counts):
    percentage = count / len(dominant_topic) * 100
    print(f"Topic {topic_id}: {count:,} reviews ({percentage:.2f}%)")

Topic 0: 17,457 reviews (13.35%)
Topic 1: 14,340 reviews (10.97%)
Topic 2: 26,794 reviews (20.49%)
Topic 3: 12,435 reviews (9.51%)
Topic 4: 13,209 reviews (10.10%)
Topic 5: 13,898 reviews (10.63%)
Topic 6: 20,477 reviews (15.66%)
Topic 7: 12,128 reviews (9.28%)


In [24]:
# Copy dữ liệu NLP để giữ nguyên checkpoint gốc
topic_df = complaints_nlp.copy()

# Dominant topic và xác suất của dominant topic
topic_df["dominant_topic"] = dominant_topic
topic_df["topic_probability"] = topic_probability

# Lưu xác suất của cả 8 topics
for topic_id in range(lda_final.n_components):
    topic_df[f"topic_{topic_id}_prob"] = doc_topic[:, topic_id]

print("Shape:", topic_df.shape)

display(
    topic_df[
        [
            "asin",
            "year",
            "dominant_topic",
            "topic_probability",
            "topic_0_prob",
            "topic_1_prob",
            "topic_2_prob",
            "topic_3_prob",
            "topic_4_prob",
            "topic_5_prob",
            "topic_6_prob",
            "topic_7_prob",
        ]
    ].head()
)

Shape: (130738, 18)


,asin,year,dominant_topic,topic_probability,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob
3,7508492919,2014,6,0.554145,0.010421,0.010422,0.010424,0.383325,0.010423,0.010421,0.554145,0.010420
5,7508492919,2014,6,0.461288,0.005004,0.373378,0.005009,0.140312,0.005003,0.005003,0.461288,0.005004
8,7508492919,2013,6,0.487617,0.003574,0.136503,0.003574,0.358009,0.003575,0.003574,0.487617,0.003574
12,7508492919,2013,6,0.446231,0.006588,0.373891,0.006591,0.006583,0.006583,0.006583,0.446231,0.146951
17,7508492919,2013,6,0.432348,0.005956,0.203788,0.334075,0.005959,0.005957,0.005957,0.432348,0.005961


In [25]:
output_path = (
    project_root
    / "data"
    / "processed"
    / "complaints_topic_model.pkl"
)

topic_df.to_pickle(output_path)

print("Saved:", output_path)
print("Shape:", topic_df.shape)

Saved: d:\data mining\e_waste_topic_modeling\data\processed\complaints_topic_model.pkl
Shape: (130738, 18)


In [26]:
topic_prob_cols = [
    f"topic_{i}_prob"
    for i in range(lda_final.n_components)
]

topic_prevalence = (
    topic_df
    .groupby("year")[topic_prob_cols]
    .mean()
    .reset_index()
)

print("Shape:", topic_prevalence.shape)

display(topic_prevalence)

Shape: (16, 9)


,year,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob
0,2003,0.001542,0.001543,0.048391,0.193120,0.726140,0.001544,0.001542,0.026177
1,2004,0.001361,0.021838,0.009886,0.028262,0.847048,0.008055,0.040471,0.043079
2,2005,0.004801,0.027091,0.024041,0.055354,0.736644,0.050122,0.047203,0.054743
3,2006,0.021295,0.053527,0.032000,0.077867,0.659347,0.044136,0.040690,0.071137
4,2007,0.009516,0.054601,0.039763,0.110194,0.576848,0.074306,0.056017,0.078755
5,2008,0.026544,0.042711,0.072250,0.106717,0.523279,0.055136,0.072578,0.100785
6,2009,0.082785,0.067358,0.097557,0.127720,0.351149,0.082136,0.098167,0.093128
7,2010,0.105422,0.078773,0.123458,0.139652,0.266062,0.083606,0.085115,0.117912
8,2011,0.104730,0.068181,0.149379,0.136658,0.214044,0.106097,0.125959,0.094952
9,2012,0.102502,0.078839,0.189553,0.136073,0.160913,0.096738,0.141145,0.094236


In [27]:
year_counts = (
    topic_df
    .groupby("year")
    .size()
    .reset_index(name="review_count")
)

topic_prevalence_with_count = topic_prevalence.merge(
    year_counts,
    on="year",
    how="left"
)

display(topic_prevalence_with_count)

,year,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,review_count
0,2003,0.001542,0.001543,0.048391,0.193120,0.726140,0.001544,0.001542,0.026177,3
1,2004,0.001361,0.021838,0.009886,0.028262,0.847048,0.008055,0.040471,0.043079,8
2,2005,0.004801,0.027091,0.024041,0.055354,0.736644,0.050122,0.047203,0.054743,26
3,2006,0.021295,0.053527,0.032000,0.077867,0.659347,0.044136,0.040690,0.071137,83
4,2007,0.009516,0.054601,0.039763,0.110194,0.576848,0.074306,0.056017,0.078755,119
5,2008,0.026544,0.042711,0.072250,0.106717,0.523279,0.055136,0.072578,0.100785,239
6,2009,0.082785,0.067358,0.097557,0.127720,0.351149,0.082136,0.098167,0.093128,369
7,2010,0.105422,0.078773,0.123458,0.139652,0.266062,0.083606,0.085115,0.117912,608
8,2011,0.104730,0.068181,0.149379,0.136658,0.214044,0.106097,0.125959,0.094952,1461
9,2012,0.102502,0.078839,0.189553,0.136073,0.160913,0.096738,0.141145,0.094236,3967


In [29]:
topic_df["review_date"] = pd.to_datetime(
    topic_df["review_date"],
    errors="coerce"
)

print(topic_df["review_date"].dtype)
print("Missing review_date:", topic_df["review_date"].isna().sum())

datetime64[us]
Missing review_date: 0


In [30]:
topic_df["review_month"] = topic_df["review_date"].dt.to_period("M")

monthly_counts = (
    topic_df
    .groupby("review_month")
    .size()
    .reset_index(name="review_count")
)

monthly_counts["review_month"] = monthly_counts["review_month"].astype(str)

display(monthly_counts)

,review_month,review_count
0,2003-07,2
1,2003-12,1
2,2004-03,1
3,2004-07,1
4,2004-08,1
...,...,...
165,2018-06,480
166,2018-07,497
167,2018-08,377
168,2018-09,120


In [31]:
monthly_counts["year"] = (
    monthly_counts["review_month"]
    .str[:4]
    .astype(int)
)

year_month_summary = (
    monthly_counts
    .groupby("year")
    .agg(
        months_with_reviews=("review_month", "count"),
        total_reviews=("review_count", "sum"),
        min_monthly_reviews=("review_count", "min"),
        max_monthly_reviews=("review_count", "max"),
        avg_monthly_reviews=("review_count", "mean")
    )
    .reset_index()
)

display(year_month_summary)

,year,months_with_reviews,total_reviews,min_monthly_reviews,max_monthly_reviews,avg_monthly_reviews
0,2003,2,3,1,2,1.500000
1,2004,6,8,1,2,1.333333
2,2005,8,26,1,6,3.250000
3,2006,12,83,2,14,6.916667
4,2007,12,119,7,16,9.916667
5,2008,12,239,9,35,19.916667
6,2009,12,369,19,45,30.750000
7,2010,12,608,31,72,50.666667
8,2011,12,1461,62,210,121.750000
9,2012,12,3967,216,863,330.583333


In [32]:
topic_prob_cols = [
    f"topic_{i}_prob"
    for i in range(lda_final.n_components)
]

monthly_topic_prevalence = (
    topic_df[
        (topic_df["year"] >= 2012) &
        (topic_df["year"] <= 2018)
    ]
    .groupby("review_month")[topic_prob_cols]
    .mean()
    .reset_index()
)

monthly_review_counts = (
    topic_df[
        (topic_df["year"] >= 2012) &
        (topic_df["year"] <= 2018)
    ]
    .groupby("review_month")
    .size()
    .reset_index(name="review_count")
)

monthly_topic_prevalence = monthly_topic_prevalence.merge(
    monthly_review_counts,
    on="review_month",
    how="left"
)

display(monthly_topic_prevalence)

,review_month,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,review_count
0,2012-01,0.093585,0.063902,0.175746,0.126663,0.218792,0.113825,0.110730,0.096756,216
1,2012-02,0.092701,0.082617,0.158101,0.135460,0.189534,0.127680,0.123208,0.090700,225
2,2012-03,0.110264,0.074631,0.157414,0.137102,0.180182,0.096718,0.140700,0.102990,243
3,2012-04,0.106078,0.066215,0.166059,0.113816,0.229618,0.108666,0.115028,0.094519,233
4,2012-05,0.133956,0.070029,0.149698,0.144705,0.188220,0.096539,0.118867,0.097987,264
...,...,...,...,...,...,...,...,...,...,...
77,2018-06,0.078495,0.165599,0.144876,0.106812,0.103730,0.141063,0.139225,0.120200,480
78,2018-07,0.076135,0.167715,0.152415,0.123254,0.083966,0.150896,0.125159,0.120461,497
79,2018-08,0.095098,0.159638,0.144278,0.106145,0.082540,0.132598,0.158810,0.120893,377
80,2018-09,0.071625,0.145245,0.156647,0.123311,0.073857,0.136629,0.134236,0.158450,120


In [33]:
temporal_df = monthly_topic_prevalence[
    monthly_topic_prevalence["review_month"] != "2018-10"
].copy()

temporal_df["review_month"] = pd.PeriodIndex(
    temporal_df["review_month"],
    freq="M"
)

temporal_df = temporal_df.sort_values("review_month").reset_index(drop=True)

print("Total months:", len(temporal_df))
print(
    "Train months:",
    (temporal_df["review_month"] <= pd.Period("2017-12", freq="M")).sum()
)
print(
    "Test months:",
    (
        (temporal_df["review_month"] >= pd.Period("2018-01", freq="M")) &
        (temporal_df["review_month"] <= pd.Period("2018-09", freq="M"))
    ).sum()
)

display(temporal_df.tail(12))

Total months: 81
Train months: 72
Test months: 9


,review_month,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,review_count
69,2017-10,0.123008,0.152170,0.151092,0.101703,0.088485,0.130070,0.149548,0.103923,985
70,2017-11,0.110157,0.158516,0.140334,0.104930,0.097832,0.128252,0.155055,0.104925,937
71,2017-12,0.106129,0.162287,0.152488,0.100065,0.081074,0.131070,0.155120,0.111767,829
72,2018-01,0.106799,0.157062,0.156144,0.092796,0.096358,0.126706,0.148629,0.115508,906
73,2018-02,0.108401,0.160459,0.148086,0.110368,0.090665,0.121950,0.154537,0.105534,715
74,2018-03,0.098819,0.163595,0.176376,0.097296,0.083282,0.112958,0.152352,0.115322,713
75,2018-04,0.090853,0.162327,0.149634,0.110355,0.084617,0.123379,0.157387,0.121448,672
76,2018-05,0.104361,0.146610,0.142893,0.111920,0.085767,0.138162,0.145688,0.124600,576
77,2018-06,0.078495,0.165599,0.144876,0.106812,0.103730,0.141063,0.139225,0.120200,480
78,2018-07,0.076135,0.167715,0.152415,0.123254,0.083966,0.150896,0.125159,0.120461,497


In [34]:
topic_prob_cols = [
    f"topic_{i}_prob"
    for i in range(lda_final.n_components)
]

lag_df = temporal_df.copy()

for col in topic_prob_cols:
    for lag in [1, 2, 3]:
        lag_df[f"{col}_lag{lag}"] = lag_df[col].shift(lag)

print("Shape:", lag_df.shape)

display(lag_df.head(8))

Shape: (81, 34)


,review_month,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,review_count,...,topic_4_prob_lag3,topic_5_prob_lag1,topic_5_prob_lag2,topic_5_prob_lag3,topic_6_prob_lag1,topic_6_prob_lag2,topic_6_prob_lag3,topic_7_prob_lag1,topic_7_prob_lag2,topic_7_prob_lag3
0,2012-01,0.093585,0.063902,0.175746,0.126663,0.218792,0.113825,0.110730,0.096756,216,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-02,0.092701,0.082617,0.158101,0.135460,0.189534,0.127680,0.123208,0.090700,225,...,NaN,0.113825,NaN,NaN,0.110730,NaN,NaN,0.096756,NaN,NaN
2,2012-03,0.110264,0.074631,0.157414,0.137102,0.180182,0.096718,0.140700,0.102990,243,...,NaN,0.127680,0.113825,NaN,0.123208,0.110730,NaN,0.090700,0.096756,NaN
3,2012-04,0.106078,0.066215,0.166059,0.113816,0.229618,0.108666,0.115028,0.094519,233,...,0.218792,0.096718,0.127680,0.113825,0.140700,0.123208,0.110730,0.102990,0.090700,0.096756
4,2012-05,0.133956,0.070029,0.149698,0.144705,0.188220,0.096539,0.118867,0.097987,264,...,0.189534,0.108666,0.096718,0.127680,0.115028,0.140700,0.123208,0.094519,0.102990,0.090700
5,2012-06,0.126379,0.076060,0.165879,0.135533,0.195325,0.101321,0.117401,0.082102,248,...,0.180182,0.096539,0.108666,0.096718,0.118867,0.115028,0.140700,0.097987,0.094519,0.102990
6,2012-07,0.091966,0.085101,0.191409,0.124079,0.158865,0.099146,0.137348,0.112086,293,...,0.229618,0.101321,0.096539,0.108666,0.117401,0.118867,0.115028,0.082102,0.097987,0.094519
7,2012-08,0.067933,0.086299,0.158804,0.145165,0.203946,0.125148,0.127563,0.085142,291,...,0.188220,0.099146,0.101321,0.096539,0.137348,0.117401,0.118867,0.112086,0.082102,0.097987


In [35]:

output_path = (
    project_root
    / "data"
    / "processed"
    / "monthly_topic_lag_features.pkl"
)

lag_df.to_pickle(output_path)

print("Saved:", output_path)
print("Shape:", lag_df.shape)

Saved: d:\data mining\e_waste_topic_modeling\data\processed\monthly_topic_lag_features.pkl
Shape: (81, 34)


In [39]:
# Save final LDA model and CountVectorizer for Streamlit

import joblib
from pathlib import Path

relative_path = Path("data/processed")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục data/processed/"
    )

artifact_dir = project_root / "models"
artifact_dir.mkdir(exist_ok=True)

# Save the FINAL CountVectorizer
joblib.dump(
    vectorizer,
    artifact_dir / "count_vectorizer.pkl"
)

# Save the FINAL K=8 LDA model
joblib.dump(
    lda_final,
    artifact_dir / "lda_model.pkl"
)

print("Saved successfully:")
print("-", artifact_dir / "count_vectorizer.pkl")
print("-", artifact_dir / "lda_model.pkl")

Saved successfully:
- d:\data mining\e_waste_topic_modeling\models\count_vectorizer.pkl
- d:\data mining\e_waste_topic_modeling\models\lda_model.pkl
